In [1]:
# CELDA 1: IMPORTACIONES BÁSICAS
import requests
import json
import time
from config import API_KEY
import pandas as pd
import boto3
import os
from datetime import datetime, timedelta
from tqdm import tqdm
import psycopg2

print("✅ Celda 1 ejecutada correctamente")
print(f"   - API_KEY cargada desde config.py (longitud: {len(API_KEY)} caracteres)")
print(f"   - Librerías importadas: requests, json, time, pandas, boto3, os, datetime, tqdm")

✅ Celda 1 ejecutada correctamente
   - API_KEY cargada desde config.py (longitud: 32 caracteres)
   - Librerías importadas: requests, json, time, pandas, boto3, os, datetime, tqdm


In [2]:
# CELDA 2: CONFIGURACIÓN PARA DESCARGA MASIVA HISTÓRICA (2016-2026)
# Fuente: Adaptado directamente de la reunión con el profesor Daniel

# Configuración de la API de RAWG
BASE_URL = "https://api.rawg.io/api/games"

# Parámetros para extracción masiva histórica (una sola vez)
FECHA_INICIAL = datetime(2016, 1, 1)    # Inicio del histórico (2016)
FECHA_FINAL = datetime(2026, 1, 1)      # Fin del histórico (1 de enero de 2026)
DIAS_TOTALES = (FECHA_FINAL - FECHA_INICIAL).days  # Total de días: 3,653

print("✅ Celda 2 ejecutada correctamente")
print(f"   - URL base: {BASE_URL}")
print(f"   - Rango histórico: {FECHA_INICIAL.date()} → {FECHA_FINAL.date()}")
print(f"   - Total de días: {DIAS_TOTALES} días ({DIAS_TOTALES // 365} años completos)")
print(f"   - Objetivo: Descargar TODOS los juegos disponibles en este rango histórico")
print(f"   - Guardado: Archivo local 'extraccion_historica.json'")

✅ Celda 2 ejecutada correctamente
   - URL base: https://api.rawg.io/api/games
   - Rango histórico: 2016-01-01 → 2026-01-01
   - Total de días: 3653 días (10 años completos)
   - Objetivo: Descargar TODOS los juegos disponibles en este rango histórico
   - Guardado: Archivo local 'extraccion_historica.json'


In [13]:
# CELDA 2B: CONFIGURACIÓN DE AWS
# Fuente: Adaptado de 17.AWS - S3 + 20.AWS - LAMBDA

# Configuración de AWS S3 (Data Lake)
S3_BUCKET_NAME = "rawg-data-lake-manuel-39"
S3_RAW_FOLDER = "raw/"
S3_PROCESSED_FOLDER = "processed/"
AWS_REGION = "eu-north-1"  # Región de Estocolmo

# Inicializar cliente de S3
s3_client = boto3.client('s3', region_name=AWS_REGION)

print("✅ Celda 2B ejecutada correctamente")
print(f"   - Bucket S3: {S3_BUCKET_NAME}")
print(f"   - Región AWS: {AWS_REGION}")
print(f"   - Carpetas: raw/ | processed/")

✅ Celda 2B ejecutada correctamente
   - Bucket S3: rawg-data-lake-manuel-39
   - Región AWS: eu-north-1
   - Carpetas: raw/ | processed/


In [3]:
# CELDA 3: EXTRACCIÓN MASIVA HISTÓRICA POR FECHAS (2016-2026)

# Inicializar lista para almacenar todos los resultados
resultados = []

print("🔄 Iniciando extracción masiva histórica...")
print(f"   - Rango: {FECHA_INICIAL.date()} → {FECHA_FINAL.date()}")
print(f"   - Total de días: {DIAS_TOTALES}")

# Iterar por cada día en el rango histórico
for i in tqdm(range(DIAS_TOTALES), desc="📅 Descargando datos históricos", unit="día"):
    # Calcular fecha actual en la iteración
    fecha_actual = FECHA_INICIAL + timedelta(days=i)
    fecha_str = fecha_actual.strftime("%Y-%m-%d")
    
    # Parámetros para la API de RAWG
    params = {
        "key": API_KEY,
        "page_size": 100,  # Máximo permitido por RAWG
        "dates": f"{fecha_str},{fecha_str}"
    }
    
    try:
        # Hacer petición a la API
        response = requests.get(BASE_URL, params=params, timeout=10)
        response.raise_for_status()
        
        # Parsear respuesta JSON
        data = response.json()
        
        # Añadir resultados si existen
        if "results" in data and data["results"]:
            resultados.extend(data["results"])
            
        # Rate limiting suave (respetar la API)
        time.sleep(0.1)
        
    except Exception as e:
        # Continuar con el siguiente día si hay error
        continue

print(f"\n✅ Extracción masiva histórica completada")
print(f"   - Juegos descargados: {len(resultados)}")
print(f"   - Guardando en archivo local: extraccion_historica.json")

# Guardar todos los resultados en un archivo JSON local
with open("extraccion_historica.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

print("   - Archivo guardado: extraccion_historica.json")
print("   - ✅ Listo para subir a S3 en la siguiente fase")

🔄 Iniciando extracción masiva histórica...
   - Rango: 2016-01-01 → 2026-01-01
   - Total de días: 3653


📅 Descargando datos históricos: 100%|██████████| 3653/3653 [44:41<00:00,  1.36día/s]  



✅ Extracción masiva histórica completada
   - Juegos descargados: 137197
   - Guardando en archivo local: extraccion_historica.json
   - Archivo guardado: extraccion_historica.json
   - ✅ Listo para subir a S3 en la siguiente fase


In [5]:
# CELDA 4: LEER JSON EN DATAFRAME Y EXPLORAR DATOS
# Fuente: Adaptado de la reunión con el profesor Daniel

# Leer el archivo JSON histórico en un DataFrame
df = pd.read_json("extraccion_historica.json")

print("✅ Celda 4 ejecutada correctamente")
print(f"   - DataFrame cargado con {len(df)} juegos")
print(f"   - Columnas disponibles: {len(df.columns)}")
print("\n📊 Primeras 5 filas del DataFrame:")
print("="*70)
print(df.head())
print("="*70)
print("\nℹ️  Información del DataFrame:")
print(f"   - Total de filas: {len(df)}")
print(f"   - Total de columnas: {len(df.columns)}")
print(f"   - Columnas principales: id, name, released, rating, metacritic, playtime")
df

✅ Celda 4 ejecutada correctamente
   - DataFrame cargado con 137197 juegos
   - Columnas disponibles: 31

📊 Primeras 5 filas del DataFrame:
                                 slug                                name  \
0  princess-remedy-in-a-world-of-hurt  Princess Remedy in a World of Hurt   
1                                echo                                ECHO   
2                            murnatan                            Murnatan   
3                    objects-in-space                    Objects In Space   
4                       rainbow-skies                       Rainbow Skies   

   playtime                                          platforms  \
0         1  [{'platform': {'id': 4, 'name': 'PC', 'slug': ...   
1         2  [{'platform': {'id': 4, 'name': 'PC', 'slug': ...   
2         1  [{'platform': {'id': 4, 'name': 'PC', 'slug': ...   
3         2  [{'platform': {'id': 4, 'name': 'PC', 'slug': ...   
4         0  [{'platform': {'id': 18, 'name': 'PlayStation ...   



,slug,name,playtime,platforms,stores,released,tba,background_image,rating,rating_top,...,tags,esrb_rating,user_game,reviews_count,saturated_color,dominant_color,short_screenshots,parent_platforms,genres,community_rating
0,princess-remedy-in-a-world-of-hurt,Princess Remedy in a World of Hurt,1,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'store': {'id': 1, 'name': 'Steam', 'slug': ...",2016-01-01,False,https://media.rawg.io/media/screenshots/138/13...,2.85,4,...,"[{'id': 31, 'name': 'Singleplayer', 'slug': 's...",None,NaN,34,0f0f0f,0f0f0f,"[{'id': -1, 'image': 'https://media.rawg.io/me...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 51, 'name': 'Indie', 'slug': 'indie'},...",NaN
1,echo,ECHO,2,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'store': {'id': 1, 'name': 'Steam', 'slug': ...",2016-01-01,False,https://media.rawg.io/media/games/584/58478387...,3.60,4,...,"[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature', ...",NaN,63,0f0f0f,0f0f0f,"[{'id': -1, 'image': 'https://media.rawg.io/me...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 3, 'name': 'Adventure', 'slug': 'adven...",NaN
2,murnatan,Murnatan,1,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'store': {'id': 1, 'name': 'Steam', 'slug': ...",2016-01-01,False,https://media.rawg.io/media/screenshots/049/04...,0.00,0,...,"[{'id': 42417, 'name': 'Экшен', 'slug': 'ekshe...",None,NaN,1,0f0f0f,0f0f0f,"[{'id': -1, 'image': 'https://media.rawg.io/me...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 2, 'name': 'Shooter', 'slug': 'shooter...",0.0
3,objects-in-space,Objects In Space,2,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'store': {'id': 1, 'name': 'Steam', 'slug': ...",2016-01-01,False,https://media.rawg.io/media/screenshots/6ac/6a...,3.57,4,...,"[{'id': 31, 'name': 'Singleplayer', 'slug': 's...",None,NaN,7,0f0f0f,0f0f0f,"[{'id': -1, 'image': 'https://media.rawg.io/me...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 3, 'name': 'Adventure', 'slug': 'adven...",NaN
4,rainbow-skies,Rainbow Skies,0,"[{'platform': {'id': 18, 'name': 'PlayStation ...","[{'store': {'id': 3, 'name': 'PlayStation Stor...",2016-01-01,False,https://media.rawg.io/media/screenshots/366/36...,0.00,0,...,[],"{'id': 3, 'name': 'Teen', 'slug': 'teen', 'nam...",NaN,4,0f0f0f,0f0f0f,"[{'id': -1, 'image': 'https://media.rawg.io/me...","[{'platform': {'id': 2, 'name': 'PlayStation',...","[{'id': 10, 'name': 'Strategy', 'slug': 'strat...",0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137192,mouse-3,MОUSE,0,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...",None,2025-12-31,False,https://media.rawg.io/media/screenshots/f47/f4...,3.17,4,...,[],None,NaN,6,0f0f0f,0f0f0f,"[{'id': -1, 'image': 'https://media.rawg.io/me...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 2, 'name': 'Shooter', 'slug': 'shooter...",NaN
137193,cairn-2025,Cairn (2025),0,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'store': {'id': 1, 'name': 'Steam', 'slug': ...",2025-12-31,False,https://media.rawg.io/media/games/147/1474adaa...,0.00,0,...,[],None,NaN,1,0f0f0f,0f0f0f,"[{'id': -1, 'image': 'https://media.rawg.io/me...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 51, 'name': 'Indie', 'slug': 'indie'},...",0.0
137194,ai-games,AI Games,0,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'store': {'id': 1, 'name': 'Steam', 'slug': ...",2025-12-31,False,https://media.rawg.io/media/screenshots/312/31...,0.00,0,...,"[{'id': 31, 'name': 'Singleplayer', 'slug': 's...",None,NaN,0,0f0f0f,0f0f0f,"[{'id': -1, 'image': 'https://media.rawg.io/me...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 40, 'name': 'Casual', 'slug': 'casual'...",0.0
137195,fnaf-free,FNAF Free,0,None,None,2025-12-31,False,None,0.00,0,...,None,None,NaN,0,0f0f0f,0f0f0f,None,NaN,"[{'id': 4, 'name': 'Action', 'slug': 'action'}]",0.0


In [7]:
# CELDA 5: CREAR BASE DE DATOS LOCAL EN POSTGRESQL
# Fuente: Adaptado de 12.SQL_en_python

import psycopg2

# Configuración de conexión local
DB_HOST = "localhost"
DB_NAME = "rawg_games_db"
DB_USER = "postgres"
DB_PASSWORD = "root"
DB_PORT = "5432"

def create_database():
    """Crear la base de datos si no existe"""
    try:
        # Conectar a PostgreSQL sin especificar base de datos
        conn = psycopg2.connect(
            host=DB_HOST,
            user=DB_USER,
            password=DB_PASSWORD,
            port=DB_PORT
        )
        conn.autocommit = True
        cursor = conn.cursor()
        
        # Verificar si la base de datos existe
        cursor.execute("SELECT 1 FROM pg_catalog.pg_database WHERE datname = %s", (DB_NAME,))
        exists = cursor.fetchone()
        
        if not exists:
            cursor.execute(f"CREATE DATABASE {DB_NAME}")
            print(f"✅ Base de datos '{DB_NAME}' creada exitosamente")
        else:
            print(f"ℹ️  Base de datos '{DB_NAME}' ya existe")
            
        cursor.close()
        conn.close()
        
    except Exception as e:
        print(f"❌ Error al crear base de datos: {e}")
        raise

def execute_sql_script():
    """Ejecutar el script SQL optimizado"""
    try:
        # Conectar a la base de datos específica
        conn = psycopg2.connect(
            host=DB_HOST,
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            port=DB_PORT
        )
        cursor = conn.cursor()
        
        # Leer y ejecutar el script SQL
        with open("01_create_rawg_database_optimized.sql", "r", encoding="utf-8") as f:
            sql_script = f.read()
        
        cursor.execute(sql_script)
        conn.commit()
        
        print("✅ Script SQL ejecutado exitosamente")
        print("   - Tablas creadas: games, genres, platforms, esrb_ratings")
        print("   - Relaciones creadas: game_genres, game_platforms")
        print("   - Vista creada: games_for_ml")
        print("   - Función creada: calculate_success()")
        
        cursor.close()
        conn.close()
        
    except Exception as e:
        print(f"❌ Error al ejecutar script SQL: {e}")
        raise

# Ejecutar creación de base de datos y script SQL
print("🔄 Creando base de datos local...")
create_database()
execute_sql_script()

print("\n✅ Celda 5 ejecutada correctamente")
print("   - Base de datos local lista para recibir datos")
print("   - Esquema optimizado implementado")

🔄 Creando base de datos local...
✅ Base de datos 'rawg_games_db' creada exitosamente
✅ Script SQL ejecutado exitosamente
   - Tablas creadas: games, genres, platforms, esrb_ratings
   - Relaciones creadas: game_genres, game_platforms
   - Vista creada: games_for_ml
   - Función creada: calculate_success()

✅ Celda 5 ejecutada correctamente
   - Base de datos local lista para recibir datos
   - Esquema optimizado implementado


In [11]:
# CELDA 6A: LIMPIAR TABLAS ANTES DE RECARGAR DATOS
# Fuente: Buenas prácticas de desarrollo local

def clean_database_tables():
    """Limpiar todas las tablas para evitar duplicados"""
    try:
        # Conectar a la base de datos
        conn = psycopg2.connect(
            host=DB_HOST,
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            port=DB_PORT
        )
        cursor = conn.cursor()
        
        print("🧹 Limpiando tablas para evitar duplicados...")
        
        # Eliminar datos en orden inverso (respetar integridad referencial)
        cursor.execute("DELETE FROM game_platforms")
        cursor.execute("DELETE FROM game_genres")
        cursor.execute("DELETE FROM games")
        cursor.execute("DELETE FROM platforms")
        cursor.execute("DELETE FROM genres")
        
        # Confirmar transacción
        conn.commit()
        
        print("✅ Tablas limpiadas exitosamente")
        print("   - game_platforms: vaciada")
        print("   - game_genres: vaciada")
        print("   - games: vaciada")
        print("   - platforms: vaciada")
        print("   - genres: vaciada")
        
        cursor.close()
        conn.close()
        
    except Exception as e:
        print(f"❌ Error al limpiar tablas: {e}")
        raise

# Ejecutar limpieza
clean_database_tables()

print("\n✅ Celda 6A ejecutada correctamente")
print("   - Base de datos lista para recarga limpia")

🧹 Limpiando tablas para evitar duplicados...
✅ Tablas limpiadas exitosamente
   - game_platforms: vaciada
   - game_genres: vaciada
   - games: vaciada
   - platforms: vaciada
   - genres: vaciada

✅ Celda 6A ejecutada correctamente
   - Base de datos lista para recarga limpia


In [12]:
# CELDA 6: TRANSFORMAR Y CARGAR DATOS DEL JSON A POSTGRESQL
# Fuente: Adaptado de 12.SQL_en_python.txt

def safe_int(value, default=0, max_val=2147483647):
    """Convertir valor a entero seguro"""
    try:
        if value is None:
            return default
        val = int(float(value))
        return min(max(val, 0), max_val)
    except (ValueError, TypeError):
        return default

def safe_numeric(value, default=0.0, max_val=9.99):
    """Convertir valor a numérico seguro"""
    try:
        if value is None:
            return default
        val = float(value)
        return min(max(val, 0.0), max_val)
    except (ValueError, TypeError):
        return default

def load_data_to_postgresql():
    """Cargar datos del DataFrame a PostgreSQL"""
    try:
        # Conectar a la base de datos
        conn = psycopg2.connect(
            host=DB_HOST,
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            port=DB_PORT
        )
        cursor = conn.cursor()
        
        print("🔄 Iniciando carga de datos a PostgreSQL...")
        print(f"   - Total de juegos a cargar: {len(df)}")
        
        # Preparar listas para inserción masiva
        games_data = []
        genres_data = set()
        platforms_data = set()
        game_genres_data = []
        game_platforms_data = []
        
        # Procesar cada juego
        for idx, row in df.iterrows():
            # Extraer added_by_status y esrb_rating de forma segura
            added_status = row.get('added_by_status') or {}
            esrb_rating = row.get('esrb_rating') or {}
            
            # Datos del juego principal con validación
            games_data.append((
                safe_int(row['id']),
                str(row['name']) if row['name'] else "Unknown",
                row['released'],
                safe_numeric(row['rating'], max_val=5.0),
                safe_int(row['ratings_count']),
                safe_int(row['metacritic'], max_val=100),
                safe_int(row['playtime']),
                safe_int(added_status.get('yet', 0)),
                safe_int(added_status.get('owned', 0)),
                safe_int(added_status.get('beaten', 0)),
                safe_int(added_status.get('toplay', 0)),
                safe_int(added_status.get('dropped', 0)),
                safe_int(added_status.get('playing', 0)),
                safe_int(esrb_rating.get('id')) if esrb_rating else None
            ))
            
            # Datos de géneros
            if 'genres' in row and isinstance(row['genres'], list):
                for genre in row['genres']:
                    if isinstance(genre, dict) and 'id' in genre and 'name' in genre:
                        genres_data.add((safe_int(genre['id']), str(genre['name'])))
                        game_genres_data.append((safe_int(row['id']), safe_int(genre['id'])))
            
            # Datos de plataformas (¡CORREGIDO!)
            if 'platforms' in row and isinstance(row['platforms'], list):
                for platform_entry in row['platforms']:
                    if isinstance(platform_entry, dict) and 'platform' in platform_entry:
                        plat = platform_entry['platform']
                        if isinstance(plat, dict) and 'id' in plat and 'name' in plat:
                            platforms_data.add((safe_int(plat['id']), str(plat['name'])))
                            released_at = platform_entry.get('released_at')
                            game_platforms_data.append((safe_int(row['id']), safe_int(plat['id']), released_at))
            
            # Mostrar progreso cada 10,000 registros
            if (idx + 1) % 10000 == 0:
                print(f"   • Progreso: {idx + 1}/{len(df)} juegos procesados")
        
        # Insertar datos en tablas maestras
        cursor.executemany(
            "INSERT INTO genres (id, name) VALUES (%s, %s) ON CONFLICT (id) DO NOTHING",
            list(genres_data)
        )
        
        cursor.executemany(
            "INSERT INTO platforms (id, name) VALUES (%s, %s) ON CONFLICT (id) DO NOTHING",
            list(platforms_data)
        )
        
        # Insertar datos en tabla principal
        cursor.executemany("""
            INSERT INTO games (
                id, name, released, rating, ratings_count, metacritic, playtime,
                status_yet, status_owned, status_beaten, status_toplay, status_dropped, status_playing,
                esrb_rating_id
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (id) DO UPDATE SET
                name = EXCLUDED.name,
                released = EXCLUDED.released,
                rating = EXCLUDED.rating,
                ratings_count = EXCLUDED.ratings_count,
                metacritic = EXCLUDED.metacritic,
                playtime = EXCLUDED.playtime,
                status_yet = EXCLUDED.status_yet,
                status_owned = EXCLUDED.status_owned,
                status_beaten = EXCLUDED.status_beaten,
                status_toplay = EXCLUDED.status_toplay,
                status_dropped = EXCLUDED.status_dropped,
                status_playing = EXCLUDED.status_playing,
                esrb_rating_id = EXCLUDED.esrb_rating_id
        """, games_data)
        
        # Insertar relaciones géneros
        cursor.executemany(
            "INSERT INTO game_genres (game_id, genre_id) VALUES (%s, %s) ON CONFLICT DO NOTHING",
            game_genres_data
        )
        
        # Insertar relaciones plataformas (¡AHORA SÍ FUNCIONARÁ!)
        cursor.executemany(
            "INSERT INTO game_platforms (game_id, platform_id, released_at) VALUES (%s, %s, %s) ON CONFLICT DO NOTHING",
            game_platforms_data
        )
        
        # Confirmar transacción
        conn.commit()
        
        print(f"\n✅ Carga completada exitosamente")
        print(f"   - Juegos insertados: {len(games_data)}")
        print(f"   - Géneros insertados: {len(genres_data)}")
        print(f"   - Plataformas insertadas: {len(platforms_data)}")
        print(f"   - Relaciones géneros: {len(game_genres_data)}")
        print(f"   - Relaciones plataformas: {len(game_platforms_data)}")  # ¡Ahora será > 0!
        
        cursor.close()
        conn.close()
        
    except Exception as e:
        print(f"❌ Error al cargar datos: {e}")
        raise

# Ejecutar carga de datos
load_data_to_postgresql()

print("\n✅ Celda 6 ejecutada correctamente")
print("   - Datos históricos cargados en base de datos local")
print("   - Listos para calcular métrica de éxito")

🔄 Iniciando carga de datos a PostgreSQL...
   - Total de juegos a cargar: 137197
   • Progreso: 10000/137197 juegos procesados
   • Progreso: 20000/137197 juegos procesados
   • Progreso: 30000/137197 juegos procesados
   • Progreso: 40000/137197 juegos procesados
   • Progreso: 50000/137197 juegos procesados
   • Progreso: 60000/137197 juegos procesados
   • Progreso: 70000/137197 juegos procesados
   • Progreso: 80000/137197 juegos procesados
   • Progreso: 90000/137197 juegos procesados
   • Progreso: 100000/137197 juegos procesados
   • Progreso: 110000/137197 juegos procesados
   • Progreso: 120000/137197 juegos procesados
   • Progreso: 130000/137197 juegos procesados

✅ Carga completada exitosamente
   - Juegos insertados: 137197
   - Géneros insertados: 19
   - Plataformas insertadas: 38
   - Relaciones géneros: 245550
   - Relaciones plataformas: 197641

✅ Celda 6 ejecutada correctamente
   - Datos históricos cargados en base de datos local
   - Listos para calcular métrica de

In [15]:
# CELDA 7: EXPORTAR DATOS LOCALES A S3
# Fuente: Adaptado de 17.AWS - S3

# 1. Cadena de conexión para pandas
db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# 2. Exportar tabla 'games' a JSON
df_games = pd.read_sql("SELECT * FROM games", db_url)
df_games.to_json("games_final.json", orient="records", indent=2, force_ascii=False)

# 3. Subir a S3 (bucket rawg-data-lake-manuel-39, carpeta processed/)
s3_client.upload_file(
    "games_final.json",
    S3_BUCKET_NAME,
    "processed/games_final_20260201.json",
    ExtraArgs={"ContentType": "application/json"}
)

print("✅ Celda 7 ejecutada correctamente")
print("   - Archivo exportado: games_final.json")
print("   - Subido a: s3://rawg-data-lake-manuel-39/processed/games_final_20260201.json")

✅ Celda 7 ejecutada correctamente
   - Archivo exportado: games_final.json
   - Subido a: s3://rawg-data-lake-manuel-39/processed/games_final_20260201.json
